# 准备输入数据
使用 TensorFlow 构建模型的第一步是将数据整理成易于使用的格式。
对 TensorFlow 而言，“易于使用”意味着数据以矩阵形式存储，并且可以快速从磁盘读取。
下面我们将演示如何实现这一点。


In [1]:
%matplotlib inline
from typing import List, Tuple
from matplotlib import pyplot as plt
from mpnn.data import make_tfrecord
from sklearn.model_selection import train_test_split
from rdkit import Chem
from tqdm import tqdm
import tensorflow as tf
import pandas as pd
import numpy as np
import json

## 获取数据
数据存储在先前项目的 [GitHub 页面](https://github.com/globus-labs/g4mp2-atomization-energy) 上


In [2]:
data = pd.read_json('../datasets/qm9.json.gz', lines=True)
print(f'Loaded {len(data)} training entries')

Loaded 25000 training entries


将 SMILES 转换为 RDKit 分子。

确保在其中添加氢原子。


In [3]:
%%time
data['mol'] = data['smiles_0'].apply(Chem.MolFromSmiles).apply(Chem.AddHs)

CPU times: user 1.14 s, sys: 38.8 ms, total: 1.18 s
Wall time: 1.18 s


## 将分子记录转换为数组字典
尽管 RDKit 分子很方便，但 TensorFlow 处理的是数值张量。接下来的几个单元格展示了如何将 RDKit 分子转换为相应格式。

第一步是将原子和键的类型转换为数值。为此，我们需要找出所有类型的原子和键。查看 [./mpnn/data.py](./mpnn/data.py) 以更好地了解该函数的作用。


In [4]:
def make_type_lookup_tables(mols: List[Chem.Mol]) -> Tuple[List[int], List[str]]:
    """Create lists of observed atom and bond types

    Args:
        mols: List of molecules used for our training set
    Returns:
        - List of atom types (elements)
        - List of bond types (elements)
    """

    # Initialize the lists
    atom_types = set()
    bond_types = set()

    # Get all types observed in these graphs
    for mol in mols:
        atom_types.update([x.GetAtomicNum() for x in mol.GetAtoms()])
        bond_types.update([x.GetBondType() for x in mol.GetBonds()])

    # Return as sorted lists
    return sorted(atom_types), sorted(bond_types)

In [5]:
atom_types, bond_types = make_type_lookup_tables(data['mol'])
print(f'Found {len(atom_types)} types of atoms: {atom_types}')
print(f'Found {len(bond_types)} types of bonds: {bond_types}')

Found 5 types of atoms: [1, 6, 7, 8, 9]
Found 4 types of bonds: [rdkit.Chem.rdchem.BondType.SINGLE, rdkit.Chem.rdchem.BondType.DOUBLE, rdkit.Chem.rdchem.BondType.TRIPLE, rdkit.Chem.rdchem.BondType.AROMATIC]


下一步是将分子转换为字典。我们需要存储分子中每个原子的类型、键的类型，以及哪些键连接哪些原子。


In [6]:
def convert_mol_to_dict(mol: Chem.Mol, atom_types: List[int], bond_types: List[str]) -> dict:
    """Convert RDKit representation of a molecule to an MPNN-ready dict
    
    Args:
        mol: Molecule to be converted
        atom_types: Lookup table of observed atom types
        bond_types: Lookup table of observed bond types
    Returns:
        (dict) Molecule as a dict
    """

    # Get the atom types, look them up in the atom_type list
    atom_type = [a.GetAtomicNum() for a in mol.GetAtoms()]
    atom_type_id = list(map(atom_types.index, atom_type))

    # Get the bond types and which atoms these connect
    connectivity = []
    edge_type = []
    for bond in mol.GetBonds():
        # Get information about the bond
        a = bond.GetBeginAtomIdx()
        b = bond.GetEndAtomIdx()
        b_type = bond.GetBondType()
        
        # Store how they are connected
        connectivity.append([a, b])
        connectivity.append([b, a])
        edge_type.append(b_type)
        edge_type.append(b_type)
    edge_type_id = list(map(bond_types.index, edge_type))

    # Sort connectivity array by the first column
    #  This is needed for the MPNN code to efficiently group messages for
    #  each atom when performing the message passing step
    connectivity = np.array(connectivity)
    if connectivity.size > 0:
        # Skip a special case of a molecule w/o bonds
        inds = np.lexsort((connectivity[:, 1], connectivity[:, 0]))
        connectivity = connectivity[inds, :]

        # Tensorflow's "segment_sum" will cause problems if the last atom
        #  is not bonded because it returns an array
        if connectivity.max() != len(atom_type) - 1:
            smiles = convert_nx_to_smiles(graph)
            raise ValueError(f"Problem with unconnected atoms for {smiles}")
    else:
        connectivity = np.zeros((0, 2))

    return {
        'n_atom': len(atom_type),
        'n_bond': len(edge_type),
        'atom': atom_type_id,
        'bond': edge_type_id,
        'connectivity': connectivity
    }

让我们以甲烷为例。


In [7]:
convert_mol_to_dict(Chem.AddHs(Chem.MolFromSmiles('C')), atom_types, bond_types)

{'n_atom': 5,
 'n_bond': 8,
 'atom': [1, 0, 0, 0, 0],
 'bond': [0, 0, 0, 0, 0, 0, 0, 0],
 'connectivity': array([[0, 1],
        [0, 2],
        [0, 3],
        [0, 4],
        [1, 0],
        [2, 0],
        [3, 0],
        [4, 0]])}

- 甲烷有 5 个原子
- 共有 4 条键（若双向计数则为 8 条）
- 第一个原子是碳（在我们的查找表中类型为 1）
- 其余原子为氢（在我们的查找表中类型为 0）
- 所有键均为单键（在我们的查找表中类型为 0）
- 所有键的起点或终点均为碳原子（原子编号 0）

看起来运行正确，现在让我们在完整数据集上运行


In [8]:
data['dict'] = data['mol'].apply(lambda x: convert_mol_to_dict(x, atom_types, bond_types))

## 将数据保存为 TFRecords
TensorFlow 有一种推荐的数据格式，称为 [`TFRecord`](https://www.tensorflow.org/tutorials/load_data/tfrecord)，它以二进制形式存储数据，读取速度非常快。虽然其细节对本教程来说稍显高级，但简而言之，我们必须先将数据转换成这种二进制格式，然后再保存到一种特殊的归档格式中。

`make_tfrecord` 函数会接收一个字典，并将其以二进制格式存储。你可能会在这种二进制格式里看到一些熟悉的词，比如 `n_bond`，但大部分内容都是人类无法直接阅读的格式。


In [ ]:
make_tfrecord(data['dict'].iloc[0])

b'\n\xf4\x01\n\x0f\n\x06n_bond\x12\x05\x1a\x03\n\x01.\n\x0f\n\x06n_atom\x12\x05\x1a\x03\n\x01\x16\np\n\x0cconnectivity\x12`\x1a^\n\\\x00\x01\x00\t\x00\n\x00\x0b\x01\x00\x01\x02\x01\x05\x02\x01\x02\x03\x02\x06\x03\x02\x03\x04\x03\x0c\x03\r\x04\x03\x04\x05\x04\x0e\x04\x0f\x05\x01\x05\x04\x05\x10\x05\x11\x06\x02\x06\x07\x06\x08\x06\x12\x07\x06\x07\x08\x07\x13\x07\x14\x08\x06\x08\x07\x08\x15\t\x00\n\x00\x0b\x00\x0c\x03\r\x03\x0e\x04\x0f\x04\x10\x05\x11\x05\x12\x06\x13\x07\x14\x07\x15\x08\n"\n\x04atom\x12\x1a\x1a\x18\n\x16\x01\x01\x01\x01\x01\x01\x01\x01\x02\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\n:\n\x04bond\x122\x1a0\n.\x00\x00\x01\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'

In [ ]:
train_data, test_data = train_test_split(data, shuffle=True, train_size=0.9)

In [ ]:
train_data, valid_data = train_test_split(train_data, train_size=0.9)

将数据以 TFDataset 格式保存到 "protobuf" 文件中


In [ ]:
for name, dataset in zip(['train', 'valid', 'test'], [train_data, valid_data, test_data]):
    # Open the file in which to store the data
    with tf.io.TFRecordWriter(f'datasets/{name}_data.proto') as writer:
        # Loop over each entry in the dataset
        for _, entry in tqdm(dataset.iterrows(), desc=name):
            # Store some output values in the dictionary as well
            record = entry['dict']
            for o in ['u0_atom', 'bandgap']:
                record[o] = entry[o]
            writer.write(make_tfrecord(record))

train: 20250it [00:02, 8253.40it/s]
valid: 2250it [00:00, 7943.84it/s]
test: 2500it [00:00, 8139.57it/s]


太好了！我们现在可以开始训练 MPNN 了。注意，我们的测试集占整个数据集的 10%（2500 条记录），训练集则是剩余 90% 数据中的 90%（20250 条记录）。
